This notebook generates some simulation data to be used in an accompanying notebook.  
  
You'll need the following packages installed:  
* `pyuvdata`: [github](https://github.com/RadioAstronomySoftwareGroup/pyuvdata/) [readthedocs](https://pyuvdata.readthedocs.io/en/latest/index.html)
    * Install with `conda install -c conda-forge pyuvdata`
* `pyuvsim`: [github](https://github.com/RadioAstronomySoftwareGroup/pyuvsim) [readthedocs](https://pyuvsim.readthedocs.io/en/latest/)
    * Install with `pip install pyuvsim`
* `pyradiosky`: [github](https://github.com/RadioAstronomySoftwareGroup/pyradiosky) [readthedocs](https://pyradiosky.readthedocs.io/en/latest/)
    * Install with `pip install pyradiosky`
* `matvis`: [github](https://github.com/HERA-Team/matvis) [readthedocs](https://matvis.readthedocs.io/en/latest/)
    * Install with `pip install matvis`
* `hera_sim`: [github](https://github.com/HERA-Team/hera_sim) [readthedocs](https://hera-sim.readthedocs.io/en/latest/)
    * Install with `pip install hera_sim`

In [1]:
import pyuvdata
import pyuvsim
import pyradiosky
import matvis
import hera_sim


/opt/conda/lib/python3.13/site-packages/pyuvdata/analytic_beam.py:174: UserWarning: basis_vector_type was not defined, defaulting to azimuth and zenith_angle.
  warnings.warn(


In [2]:
import numpy as np
import matplotlib.pyplot as plt

import hera_sim  # For running simulations
from astropy import constants, units  # For unit conversions, etc
from astropy.coordinates import Latitude, Longitude
from pyuvdata import UVData, AiryBeam  # For interfacing with data
from pyradiosky import SkyModel  # For managing what's on the sky
%matplotlib inline

In [3]:
plt.rcParams["figure.facecolor"] = "w"
plt.rcParams["font.family"] = "serif"

In [4]:
# array_layout = hera_sim.antpos.hex_array(
#     2, split_core=False, outriggers=0
# )  # 7-element hexagonal array
diameter = 14  # HERA dish diameter
sep = 3 * diameter
array_layout = {i: np.array([i%2, i//2, 0]) * sep for i in range(4)}
# start_freq = 110058593.75
start_freq = 100e6
Nfreqs = 250
# channel_width = 97656.25
channel_width = 1e6
start_time = 2458116.4763345513
integration_time = 20.0
Ntimes = int(2 * 3600 / integration_time)
polarization_array = [-5,]
uvdata = hera_sim.io.empty_uvdata(
    array_layout=array_layout,
    Nfreqs=Nfreqs,
    start_freq=start_freq,
    channel_width=channel_width,
    Ntimes=Ntimes,
    start_time=start_time,
    integration_time=integration_time,
    polarization_array=polarization_array,
    x_orientation="east",
    blt_order=("time", "ant1"),
)

No beam information, so cannot determine telescope mount_type, feed_array or feed_angle. Specify a telescope config file in the obs param file to get beam information or, if calling from `initialize_uvdata_from_keywords`, specify mount_type, feed_array and feed_angle.
The default baseline conjugation convention has changed. In the past it was 'ant2<ant1', it now defaults to 'ant1<ant2'. You can specify the baseline conjugation convention in `obs_param` by setting the obs_param['ordering']['conjugation_convention'] field. This warning will go away in version 1.5.
Unknown polarization basis -- assuming linearly polarized (x/y) feeds for feed_array.


In [5]:
from astropy.coordinates import Longitude, Latitude

In [6]:
transit_lst = uvdata.lst_array[np.unique(uvdata.time_array, return_index=True)[1]].mean()
src_dec = np.array([-30.721528,], dtype=float) * units.deg
src_ra = np.array([transit_lst*units.rad.to(units.deg),], dtype=float) * units.deg
src_flux = 1 * (uvdata.freq_array.flatten() / 150e6)**-2
stokes = np.zeros((4, Nfreqs, 1), dtype=float)
stokes[0,:,0] = src_flux

In [7]:
sky_model = SkyModel(
    name=np.arange(stokes.shape[-1]).astype(str),
    ra=Longitude(src_ra),
    dec=Latitude(src_dec),
    stokes=stokes * units.Jy,
    freq_array=uvdata.freq_array.flatten() * units.Hz,
    frame="icrs",
    spectral_type="subband",
)


freq_edge_array not set, calculating it from the freq_array.


In [8]:
beam = AiryBeam(diameter=14)

In [9]:
data_model = hera_sim.visibilities.ModelData(
    uvdata=uvdata,
    sky_model=sky_model,
    beam_ids=None,
    beams=[beam,],
)
simulation = hera_sim.visibilities.VisibilitySimulation(
    data_model=data_model,
    simulator=hera_sim.visibilities.MatVis(),
)
simulation.simulate();


In [10]:
uvdata.write_uvh5("single_source_example.uvh5", clobber=True, fix_autos=True)

File exists; clobbering


Parameter feed_array, feed_angle, and mount_type must be set together. Unsetting optional parameters.
